# Chapter 6 — SCMI: System Control and Management Interface

## Concept
SCMI (ARM DEN0056) provides an OS-to-firmware interface for clock, performance,
power domain, sensor, and reset management. The transport layer can be shared
memory (mailbox), SMC, or VirtIO (virtio-scmi). Clock domains expose
`rate_get` / `rate_set`; performance domains expose OPP tables.

> **Open question resolved**: This lab assumes the kernel image has
> `CONFIG_ARM_SCMI_PROTOCOL=y` and at least one SCMI transport enabled.
> Ubuntu 24.04 ARM64 cloud kernel includes `arm_scmi` as a module.

## Lab Objectives
1. Load the SCMI kernel module if not already loaded.
2. Confirm SCMI driver probes without error in `dmesg`.
3. List SCMI clock domains via `/sys/bus/platform/drivers/arm-scmi/`.
4. Verify performance domain visible via `/sys/devices/system/cpu/cpufreq/`.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# CONFIG_ARM_SCMI_PROTOCOL required in guest kernel
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=["-netdev", "user,id=net0", "-device", "virtio-net-pci,netdev=net0"],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Check SCMI module and load if needed ─────────────────────────────
module_check = sc.run_command(
    "lsmod | grep scmi || echo 'scmi not loaded'", timeout=10
)
print(module_check)

if "scmi" not in module_check or "not loaded" in module_check:
    load_out = sc.run_command(
        "sudo modprobe arm_scmi 2>&1 || echo 'modprobe failed'", timeout=15
    )
    print("modprobe arm_scmi:", load_out)
    time.sleep(2)


In [ ]:
# ── Step 2: Check dmesg for SCMI probe messages ───────────────────────────────
scmi_dmesg = sc.run_command(
    "dmesg | grep -i 'scmi\|virtio-scmi\|arm-scmi' | tail -20",
    timeout=10
)
print("SCMI dmesg lines:\n", scmi_dmesg)


In [ ]:
# ── Step 3: Enumerate SCMI devices in sysfs ──────────────────────────────────
scmi_sysfs = sc.run_command(
    "ls /sys/bus/platform/drivers/arm-scmi/ 2>/dev/null || "
    "ls /sys/bus/scmi/devices/ 2>/dev/null || "
    "echo 'SCMI sysfs path not found'",
    timeout=10
)
print("SCMI sysfs:", scmi_sysfs)


In [ ]:
# ── Step 4: List clock domains (if SCMI clock driver present) ────────────────
clk_domains = sc.run_command(
    "ls /sys/kernel/debug/clk/ 2>/dev/null | head -20 || "
    "cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_available_frequencies "
    "2>/dev/null || echo 'clock domain info not available'",
    timeout=10
)
print("Clock domains / available frequencies:\n", clk_domains)


In [ ]:
# ── Step 5: Check performance domain via cpufreq ─────────────────────────────
cpufreq = sc.run_command(
    "cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_driver 2>/dev/null || "
    "echo 'cpufreq scaling driver not found'",
    timeout=10
)
print("cpufreq scaling driver:", cpufreq)

# Check for transport timeout in dmesg
timeout_check = sc.run_command(
    "dmesg | grep -i 'timeout\|transport' | grep -i 'scmi' || echo 'none'",
    timeout=10
)
print("SCMI transport timeout lines:", timeout_check)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
# SCMI probe: either module loaded or built-in
scmi_present = ("scmi" in module_check.lower() or
                "scmi" in scmi_dmesg.lower() or
                "arm_scmi" in scmi_dmesg.lower())
assert_true(scmi_present,
            "SCMI driver present (module or built-in)",
            detail=module_check[:80],
            action="Check CONFIG_ARM_SCMI_PROTOCOL; load arm_scmi module manually")

assert_not_contains(scmi_dmesg, r"error|Error|ERROR",
                    "No SCMI probe errors in dmesg",
                    action="Inspect dmesg for SCMI transport binding errors")

# Sysfs check (informational — may not be present without SCMI firmware)
assert_true(
    "not found" not in scmi_sysfs.lower() or "cpufreq" in clk_domains.lower(),
    "SCMI or cpufreq sysfs path accessible",
    detail=scmi_sysfs[:80],
    action="QEMU virt does not expose SCMI firmware by default; "
           "virtio-scmi device required for full SCMI lab",
)

assert_not_contains(timeout_check, r"timeout",
                    "No SCMI transport timeout in dmesg",
                    action="Check virtio-scmi transport or mailbox configuration")

# cpufreq driver (any driver is acceptable; scmi-cpufreq is ideal)
assert_true(len(cpufreq.strip()) > 0 and "not found" not in cpufreq,
            "cpufreq scaling driver present",
            detail=cpufreq.strip(),
            action="Kernel may lack cpufreq driver; check CONFIG_CPUFREQ")


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| SCMI driver present | Yes | Module or built-in |
| No SCMI probe errors | Clean | Transport bind succeeds |
| SCMI/cpufreq sysfs | Accessible | Domain enumeration path |
| No transport timeout | Clean | Mailbox / VirtIO stable |
| cpufreq scaling driver | Present | Performance domain exposed |

> **Fidelity note**: QEMU `virt` does not implement a SCMI firmware agent.
> A complete SCMI lab requires either `virtio-scmi` (upstream QEMU ≥ 7.2)
> or a TF-A build with the SCMI server. The assertions above pass if the
> SCMI *kernel driver* probes correctly, even without a firmware counterpart.
> `CONFIG_ARM_SCMI_PROTOCOL` must be built-in or loaded as `arm_scmi`.

## Next Steps
→ **Chapter 7**: ACPI on ARM — dump and parse ACPI tables from the UEFI guest.
